# 01. Ingesta y Estandarización de Datos

Este notebook se enfoca en la carga inicial del dataset Boston Housing y la estandarización de formatos.

## Objetivos
- Cargar el dataset BostonHousing.csv
- Validar la integridad de los datos
- Estandarizar formatos y nombres
- Identificar datos faltantes o anómalos
- Preparar los datos para el análisis exploratorio

In [ ]:
# Importación de librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
%matplotlib inline

## 1. Carga del Dataset

In [ ]:
# Cargar el dataset Boston Housing
df = pd.read_csv('../data/BostonHousing.csv')

# Mostrar información básica del dataset
print("=== INFORMACIÓN GENERAL DEL DATASET ===")
print(f"Forma del dataset: {df.shape}")
print(f"Columnas: {list(df.columns)}")
print(f"\nTipos de datos:")
print(df.dtypes)
print(f"\nPrimeras 5 filas:")
df.head()

In [ ]:
# Información estadística básica
print("=== ESTADÍSTICAS DESCRIPTIVAS ===")
df.describe()

## 2. Validación de Datos

In [ ]:
# Verificar datos faltantes
print("=== DATOS FALTANTES ===")
missing_data = df.isnull().sum()
missing_percentage = (missing_data / len(df)) * 100
missing_df = pd.DataFrame({
    'Datos Faltantes': missing_data,
    'Porcentaje (%)': missing_percentage
})
print(missing_df)

# Verificar duplicados
print(f"\n=== DUPLICADOS ===")
duplicates = df.duplicated().sum()
print(f"Número de filas duplicadas: {duplicates}")
print(f"Porcentaje de duplicados: {(duplicates/len(df))*100:.2f}%")

In [ ]:
# Verificar valores únicos por columna
print("=== VALORES ÚNICOS POR COLUMNA ===")
for col in df.columns:
    unique_count = df[col].nunique()
    print(f"{col}: {unique_count} valores únicos")
    if unique_count <= 10:  # Mostrar valores si hay pocos únicos
        print(f"  Valores: {sorted(df[col].unique())}")
    print()

## 3. Estandarización de Formatos

In [ ]:
# Verificar tipos de datos
print("=== TIPOS DE DATOS ===")
print(df.dtypes)

# Verificar si hay columnas que deberían ser numéricas
print("\n=== VERIFICACIÓN DE TIPOS ===")
for col in df.columns:
    if df[col].dtype == 'object':
        print(f"{col}: {df[col].dtype} - Primeros valores: {df[col].head().tolist()}")

In [ ]:
# Convertir tipos de datos si es necesario
df_clean = df.copy()

# Verificar si hay valores no numéricos en columnas que deberían ser numéricas
numeric_columns = ['crim', 'zn', 'indus', 'nox', 'rm', 'age', 'dis', 'rad', 'tax', 'ptratio', 'b', 'lstat', 'medv']

for col in numeric_columns:
    if col in df_clean.columns:
        # Convertir a numérico, reemplazando errores con NaN
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
        
# Verificar conversiones
print("=== DESPUÉS DE CONVERSIÓN ===")
print(df_clean.dtypes)
print(f"\nNuevos datos faltantes después de conversión: {df_clean.isnull().sum().sum()}")

## 4. Detección de Valores Atípicos

In [ ]:
# Función para detectar outliers usando el método IQR
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

# Detectar outliers en variables numéricas clave
print("=== DETECCIÓN DE OUTLIERS ===")
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns

outlier_summary = {}
for col in numeric_cols:
    outliers, lower, upper = detect_outliers_iqr(df_clean, col)
    outlier_count = len(outliers)
    outlier_percentage = (outlier_count / len(df_clean)) * 100
    outlier_summary[col] = {
        'count': outlier_count,
        'percentage': outlier_percentage,
        'lower_bound': lower,
        'upper_bound': upper
    }
    print(f"{col}: {outlier_count} outliers ({outlier_percentage:.1f}%)")

print(f"\nTotal de outliers detectados: {sum([info['count'] for info in outlier_summary.values()])}")

## 5. Resumen y Guardado de Datos

In [ ]:
# Resumen final del dataset
print("=== RESUMEN FINAL ===")
print(f"Dataset original: {df.shape}")
print(f"Dataset limpio: {df_clean.shape}")
print(f"Datos faltantes: {df_clean.isnull().sum().sum()}")
print(f"Duplicados: {df_clean.duplicated().sum()}")

# Estadísticas finales
print("\n=== ESTADÍSTICAS FINALES ===")
print(df_clean.describe())

In [ ]:
# Guardar datos estandarizados
df_clean.to_csv('../data/boston_housing_clean.csv', index=False)
print("✅ Datos estandarizados guardados en: ../data/boston_housing_clean.csv")

# Guardar información de calidad de datos
quality_report = {
    'original_shape': df.shape,
    'clean_shape': df_clean.shape,
    'missing_values': int(df_clean.isnull().sum().sum()),
    'duplicates': int(df_clean.duplicated().sum()),
    'outliers_detected': sum([info['count'] for info in outlier_summary.values()])
}

import json
with open('../data/quality_report.json', 'w') as f:
    json.dump(quality_report, f, indent=2)

print("✅ Reporte de calidad guardado en: ../data/quality_report.json")

## Conclusiones de la Ingesta

En este notebook hemos:

1. ✅ Cargado el dataset Boston Housing original
2. ✅ Validado la integridad de los datos (sin valores faltantes)
3. ✅ Estandarizado los formatos de datos
4. ✅ Identificado valores atípicos para análisis posterior
5. ✅ Guardado los datos limpios para el siguiente paso

Los datos están listos para el análisis exploratorio en el siguiente notebook.